# FactoryIO
### modbus_ex4a 대상, 확장

In [13]:
# 모드버스접속
from pymodbus.client import ModbusTcpClient
import time as tt
import threading
client = ModbusTcpClient('210.119.14.74', port=502)
client.connect()

True

In [14]:
# 초기화
client.write_coils(0, [0]*16 )

In [15]:
# 켜야되는 출력 2부터 7개 모두 On
client.write_coils(2, [1]*7 )

In [16]:
# 클래스로 2공정 묶기

class Vscan:
    def __init__(self, client, bit1, bit2, coil):
        self.client = client
        self.scan_run = True
        self.bit1 = bit1
        self.bit2 = bit2
        self.coil = coil
        self.mem = 0

    def run(self):
        while self.scan_run:
            bits = self.client.read_discrete_inputs(0, count=8).bits
            if bits[self.bit1]:
                if self.mem == 0:
                    self.mem = tt.monotonic()
                elif tt.monotonic() - self.mem >= 1:
                    self.client.write_coil(self.coil,1) # Pusher On
                    self.mem = 0
            else:
                self.mem = 0  
            if bits[self.bit2]: 
                self.client.write_coil(self.coil,0) # Pusher Off
                self.mem = 0
            tt.sleep(0.1)

    def start(self):
        self.thread = threading.Thread(target=self.run, daemon=True)
        self.thread.start()
        print("스레드 시작")
    def stop(self):
        self.scan_run = False

In [17]:
vscan1 = Vscan(client, 0, 3, 0)
vscan2 = Vscan(client, 1, 5, 1)
vscan1.start()
vscan2.start()

스레드 시작
스레드 시작


Exception in thread Thread-5 (run):
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/opt/conda/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_3539/1231886864.py", line 14, in run
  File "/opt/conda/lib/python3.11/site-packages/pymodbus/client/mixin.py", line 91, in read_discrete_inputs
    return self.execute(no_response_expected, pdu)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pymodbus/client/base.py", line 190, in execute
    return self.transaction.sync_execute(no_response_expected, request)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pymodbus/transaction/transaction.py", line 132, in sync_execute
    response = self.sync_get_response(request.dev_id, request.transaction_id)
               ^^^^^^^^